# Module 2: Vector Search (lessons 02-06, with the lesson 09 ONNX embedder)

The course uses `SentenceTransformer("all-MiniLM-L6-v2")` (PyTorch). On this Windows machine PyTorch does not work (Python 3.15 crashes on model load; on 3.13/3.14 Smart App Control blocks `torch_python.dll`). So we use the course's own ONNX `Embedder` from lesson 09: same model, same vectors, no PyTorch.

| Course | Here |
|---|---|
| `model = SentenceTransformer("all-MiniLM-L6-v2")` | `model = Embedder()` |
| `model.encode(text)` | `model.encode(text)` (same) |
| `model.encode(list_of_texts)` | `model.encode_batch(list_of_texts)` |

Run once before this notebook: `uv run python download.py` (inside `module2-vector-search/`).

## Lesson 02: Embeddings

In [1]:
from embedder import Embedder

model = Embedder()   # models/Xenova/all-MiniLM-L6-v2 (ONNX)

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1.shape, v1[:5]

((384,),
 array([ 0.02139038, -0.07397998,  0.0014207 ,  0.02138164,  0.02451135]))

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

v1.dot(dv)

np.float64(0.32332391095966817)

In [4]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

v2.dot(dv)

np.float64(0.019730407140850777)

Vectors are normalized (length 1), so the dot product equals cosine similarity.

In [5]:
import numpy as np
np.linalg.norm(v1), np.linalg.norm(dv)

(np.float64(1.0000000000000002), np.float64(1.0))

Extra: the two paraphrased questions from lesson 01, which keyword search could not match.

In [6]:
a = model.encode("Can I still join the course after the start date?")
b = model.encode("Is it possible to enroll late?")
c = model.encode("How do I run Docker on Windows?")
print("join vs enroll late:", round(float(a.dot(b)), 3))
print("join vs docker     :", round(float(a.dot(c)), 3))

join vs enroll late: 0.521
join vs docker     : -0.107


## Lesson 03: Embedding our dataset

In [7]:
from ingest import load_faq_data

documents = load_faq_data()
len(documents)

1456

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
from tqdm.auto import tqdm
import time

batch_size = 50
vectors = []

t0 = time.time()
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode_batch(batch)   # course: model.encode(batch)
    vectors.extend(batch_vectors)
print(f"{time.time() - t0:.1f} seconds")

len(vectors)

C:\Users\vasus\Projects\agent-ai-bootcamp\module2-vector-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/30 [00:00<?, ?it/s]

  3%|▎         | 1/30 [00:00<00:28,  1.00it/s]

  7%|▋         | 2/30 [00:02<00:28,  1.02s/it]

 10%|█         | 3/30 [00:03<00:27,  1.03s/it]

...

 97%|█████████▋| 29/30 [00:35<00:01,  1.19s/it]

100%|██████████| 30/30 [00:36<00:00,  1.09it/s]

100%|██████████| 30/30 [00:36<00:00,  1.20s/it]

36.0 seconds


1456

In [10]:
X = np.array(vectors)
X.shape

(1456, 384)

## Lesson 04: Vector search with numpy

In [11]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)
scores.shape

(1456,)

In [12]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(645), np.float64(0.7629410509237896))

In [13]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [14]:
top5 = np.argsort(-scores)[:5]

for idx in top5:
    print(round(float(scores[idx]), 3), "|", documents[idx]["course"], "|", documents[idx]["question"])

0.763 | data-engineering-zoomcamp | Course: Can I still join the course after the start date?
0.758 | mlops-zoomcamp | Course - Can I still join the course after the start date?
0.719 | machine-learning-zoomcamp | The course has already started. Can I still join it?
0.654 | llm-zoomcamp | I just discovered the course. Can I still join?
0.56 | data-engineering-zoomcamp | Course - Can I follow the course after it finishes?


## Lesson 05: Vector search with minsearch

In [15]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [16]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Can I run the course locally instead of Codespaces?',
 'Does the course certificate show the number of course hours?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?']

### Extra: keyword vs vector on the paraphrased question
Module 1 text search failed on "Is it possible to enroll late?".

In [18]:
from ingest import build_index

text_index = build_index(documents)
q = "Is it possible to enroll late?"
f = {"course": "llm-zoomcamp"}

print("KEYWORD:")
for d in text_index.search(q, filter_dict=f, boost_dict={"question": 3.0, "section": 0.5}, num_results=3):
    print("  -", d["question"])
print("VECTOR:")
for d in vindex.search(model.encode(q), filter_dict=f, num_results=3):
    print("  -", d["question"])

KEYWORD:
  - Is it a group project?
  - My dataset is copyrighted and I can't publish it. Can I still use it for the project?
  - uv says Failed to hardlink files in Codespaces. Is it an error?
VECTOR:
  - I just discovered the course. Can I still join?
  - Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
  - Certificate: Can I follow the course in a self-paced mode and get a certificate?


## Lesson 06: RAG with vector search

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")   # keys live in the project root .env

# Course: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [20]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=text_index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
print(assistant.rag(query))

Yes—you can still join the course. Just keep in mind that if you want to earn a certificate you’ll need to submit your capstone project before the submission window closes.


In [21]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [22]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

print(vector_assistant.rag("the program has already begun, can I still sign up?"))

Yes—you can still join even though the program has already started. Feel free to start learning and submitting homework while the submission form remains open. If you’d like to receive a certificate, just be sure to submit your capstone project before the submission window closes.


Extra: what each assistant retrieves for the rephrased question.

In [23]:
q = "the program has already begun, can I still sign up?"
print("KEYWORD :", [d["question"][:60] for d in assistant.search(q)][:3])
print("VECTOR  :", [d["question"][:60] for d in vector_assistant.search(q)][:3])

KEYWORD : ['The homework submission form is still open even though the d', 'I just discovered the course. Can I still join?', 'I missed the first homework - can I still get a certificate?']
VECTOR  : ['I just discovered the course. Can I still join?', 'Course: I have registered for the LLM Zoomcamp. When can I e', 'Can I run the course locally instead of Codespaces?']
